# Session 2: the S-N curve as an equation, and what shifts it
**A. Real-world problem.** A shaft carries a pulley load and turns at 1500 revolutions per
minute (25 revolutions per second). Every revolution puts each surface point through one full
pull-push cycle, so about 11 hours of running make a million cycles. What diameter keeps the shaft safe for 10^6 cycles
and beyond, with a margin? On the way we meet two things that shorten life: a *mean* stress
and a *notch* such as a hole or a sharp shoulder. All numbers are illustrative and the model
is simplified; it is not for designing real parts.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

SU = 600.0     # ultimate strength, MPa (illustrative steel)
SE = 300.0     # endurance limit of a small polished test bar, MPa

## Step 1: Basquin's relation for any part
`S_a = A * N**b` through two anchors: `f*S_u` at 1000 cycles (f = 0.9) and the endurance limit
`S_e` at 10^6 cycles. Then `b = -log10(f*S_u / S_e) / 3` and `A = (f*S_u)^2 / S_e`.
A real part has a lower endurance limit than a polished test bar (rough surface, larger size);
we pass that lower value in as `se`.

In [ ]:
def basquin_constants(su=SU, se=SE, f=0.9):
    """Return (A, b) of S_a = A * N**b for ultimate strength su and endurance limit se (MPa)."""
    b = -np.log10(f * su / se) / 3.0
    a = (f * su) ** 2 / se
    return a, b


def life(sa, su=SU, se=SE):
    """Cycles to failure at fully reversed amplitude sa (MPa); inf at or below the endurance limit."""
    if sa <= se:
        return np.inf
    a, b = basquin_constants(su, se)
    return float((sa / a) ** (1.0 / b))


def stress_for_life(n, su=SU, se=SE):
    """Amplitude (MPa) that gives exactly n cycles (1e3 <= n <= 1e6 is where the line is anchored)."""
    a, b = basquin_constants(su, se)
    return float(a * n ** b)

## Step 2: mean stress and the Goodman line
A load that swings between 0 and a maximum has a mean as well as an amplitude. A tensile mean
opens cracks and shortens life. The Goodman line joins `S_e` (on the amplitude axis) and `S_u`
(on the mean axis):

    S_a / S_e + S_m / S_u = 1

A point below the line is "safe for indefinite life" in this model. To use the S-N curve with
a mean stress, convert to an equivalent fully reversed amplitude:
`S_eq = S_a / (1 - S_m / S_u)`.

In [ ]:
def goodman_sum(sa, sm, su=SU, se=SE):
    """S_a/S_e + S_m/S_u. Less than 1: indefinite life predicted; equal to 1: on the line."""
    return sa / se + sm / su


def equivalent_amplitude(sa, sm, su=SU):
    """Fully reversed amplitude with the same damage as (sa, sm), by Goodman. Needs sm < su."""
    if sm >= su:
        raise ValueError("mean stress must be below the ultimate strength")
    return sa / (1.0 - sm / su)


def life_with_mean(sa, sm, su=SU, se=SE):
    """Cycles to failure for amplitude sa and mean sm, using Goodman and Basquin."""
    return life(equivalent_amplitude(sa, sm, su), su, se)

## Step 3: a notch raises the local stress (K_t)
Near a hole or a sharp corner the stress is higher than the nominal value `S_nom = F / A_net`
by a *stress concentration factor* `K_t`, so `S_local = K_t * S_nom`. For a small hole in a
wide plate pulled in one direction the exact elastic result (Kirsch) for the stress around the
edge is `S_theta = S * (1 - 2 cos 2 theta)`, where theta is measured from the load direction.
The largest value, at theta = 90 degrees, is 3 S. So K_t = 3 for a hole in a wide plate. We
apply K_t to both amplitude and mean (a simple, conservative choice; real designs use a
smaller fatigue notch factor K_f).

In [ ]:
def hole_hoop_stress(theta_deg, r_over_a=1.0, sigma=1.0):
    """Kirsch hoop stress at polar position (r/a, theta) for a hole of radius a in a wide plate
    under uniaxial stress sigma along theta = 0."""
    th = np.radians(theta_deg)
    q = 1.0 / r_over_a ** 2
    return sigma / 2 * (1 + q) - sigma / 2 * (1 + 3 * q * q) * np.cos(2 * th)


def stress_concentration_hole():
    """K_t of a hole in a wide plate: maximum hoop stress at the edge divided by the nominal stress."""
    theta = np.linspace(0, 180, 3601)
    return float(hole_hoop_stress(theta).max())


def life_with_notch(sa_nom, sm_nom, kt, su=SU, se=SE):
    """Life when the nominal amplitude and mean are both magnified by kt."""
    return life_with_mean(kt * sa_nom, kt * sm_nom, su, se)

## Step 4: design a rotating shaft
A shaft under a steady bending moment M that turns sees a fully reversed surface stress of
amplitude `S_a = 32 M / (pi d^3)` (d in metres, M in N m, result in Pa). To survive 10^6 cycles
and beyond with a safety factor `n` on stress, require `K_t * S_a <= S_e,part / n`, so
`d = (32 M n K_t / (pi S_e,part))^(1/3)`. The part endurance limit is lower than the test bar's:
we use S_e,part = 0.7 * S_e for surface finish and size (an illustrative factor).

In [ ]:
def bending_stress_mpa(moment_nm, d_m):
    """Surface bending stress amplitude (MPa) of a round shaft of diameter d_m (m)."""
    return 32.0 * moment_nm / (np.pi * d_m ** 3) / 1e6


def design_diameter_mm(moment_nm, n=2.0, kt=1.0, se_part=0.7 * SE):
    """Smallest diameter (mm) with safety factor n on stress at the endurance limit."""
    d = (32.0 * moment_nm * n * kt / (np.pi * se_part * 1e6)) ** (1.0 / 3.0)
    return d * 1000.0


def shaft_life(moment_nm, d_mm, kt=1.0, su=SU, se_part=0.7 * SE):
    """Cycles to failure of the shaft (inf if below the part endurance limit)."""
    return life(kt * bending_stress_mpa(moment_nm, d_mm / 1000.0), su, se_part)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    a, b = basquin_constants()
    n = np.logspace(3, 7, 200)
    fig, ax = plt.subplots(figsize=(6.5, 4.4))
    ax.loglog(n, np.where(n < 1e6, a * n ** b, SE), label="Basquin line, polished bar")
    a2, b2 = basquin_constants(SU, 0.7 * SE)
    ax.loglog(n, np.where(n < 1e6, a2 * n ** b2, 0.7 * SE), "--", label="same steel, real part (S_e x 0.7)")
    ax.set_xlabel("cycles to failure N")
    ax.set_ylabel("stress amplitude S_a (MPa)")
    ax.set_title(f"Log-log S-N curve: slope b = {b:.3f} (bar), {b2:.3f} (part)")
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "sn_loglog.png", dpi=120)

    sm = np.linspace(0, SU, 100)
    fig2, ax2 = plt.subplots(figsize=(6, 4.2))
    ax2.plot(sm, SE * (1 - sm / SU), label="Goodman line")
    ax2.fill_between(sm, 0, SE * (1 - sm / SU), alpha=0.15, label="indefinite life predicted")
    ax2.plot([150], [100], "ko")
    ax2.annotate("(150, 100): sum = %.2f" % goodman_sum(100, 150), (150, 100), textcoords="offset points", xytext=(8, 8), fontsize=8)
    ax2.set_xlabel("mean stress S_m (MPa)")
    ax2.set_ylabel("stress amplitude S_a (MPa)")
    ax2.set_title("Goodman diagram")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "goodman.png", dpi=120)

    th = np.linspace(0, 180, 361)
    fig3, ax3 = plt.subplots(figsize=(6, 4.0))
    ax3.plot(th, hole_hoop_stress(th), label="edge of the hole (r = a)")
    ax3.plot(th, hole_hoop_stress(th, 2.0), label="r = 2a")
    ax3.axhline(1, color="gray", ls=":")
    ax3.set_xlabel("angle from the load direction (degrees)")
    ax3.set_ylabel("hoop stress / nominal stress")
    ax3.set_title("Hole in a wide plate: peak 3 at 90 degrees")
    ax3.grid(alpha=0.3)
    ax3.legend(fontsize=8)
    fig3.tight_layout()
    fig3.savefig(OUT / "hole_stress.png", dpi=120)

    print(f"Kt of a hole = {stress_concentration_hole():.3f}")
    print(f"b (bar) = {b:.4f}, b (part) = {b2:.4f}")
    print(f"Goodman sum at (Sa, Sm) = (100, 150): {goodman_sum(100, 150):.3f}")
    for kt in (1.0, 2.0):
        print(f"design diameter M = 200 N m, n = 2, Kt = {kt}: {design_diameter_mm(200, 2, kt):.2f} mm")
    print(f"27 mm shaft stress = {bending_stress_mpa(200, 0.027):.1f} MPa")
    print(f"20 mm shaft life = {shaft_life(200, 20):,.0f} cycles")

## Where this stops
Real endurance limits come from tests of the actual part. Aluminium has no true endurance
limit. Variable loads need counting and adding up damage: Session 3.

## Coding tasks
1. **Easy.** Use `life_with_notch` to find the nominal amplitude (zero mean) at which a plate with a hole stops having an indefinite life. Check 80 MPa and 110 MPa first.
2. **Stretch.** Add a function that returns the largest diameter reduction allowed for a shaft with K_t = 2 if the safety factor is lowered from 2 to 1.5, and print the diameters.
3. **Explain.** Explain in your own words why the Goodman line does not need a separate S-N curve for every mean stress.

## Assessment
- **A. Conceptual.** Why does a tensile mean stress shorten fatigue life, and why is the same load safe for steel but not necessarily for aluminium when it lies below the endurance limit?
- **B. Numerical.** A load has S_a = 240 MPa and S_m = 250 MPa on the illustrative steel (S_u = 600, S_e = 300). Find the Goodman equivalent amplitude and the life. What would the life be with zero mean?
- **C. Graph and simulation interpretation.** On the Goodman diagram the point (S_m, S_a) = (300, 150) MPa is plotted. Where is it, and what does the Goodman sum say? What happens if the amplitude rises slightly?
- **D. Design and optimisation challenge.** Design: a shaft carries M = 150 N m, has a shoulder fillet with K_t = 1.5, and must have a safety factor of 2 on stress for indefinite life (part endurance limit 210 MPa). Choose a diameter, compare 25 mm and 28 mm, and say what happens at 22 mm. State the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    a, b = L.basquin_constants()
    checks.append(("Basquin anchors: 540 MPa at 1000 cycles and 300 MPa at 1e6 cycles",
                   abs(a * 1e3 ** b - 540) < 1e-6 and abs(a * 1e6 ** b - 300) < 1e-6))
    checks.append((f"b is about -0.085 for the bar (got {b:.4f}) and about -0.137 for the part (S_e = 210)",
                   abs(b + 0.0851) < 1e-4 and abs(L.basquin_constants(600, 210)[1] + 0.1367) < 1e-4))
    checks.append(("stress_for_life and life are inverse functions", abs(L.life(L.stress_for_life(1e5)) / 1e5 - 1) < 1e-9))

    # Goodman line, verified numerically
    sms = np.linspace(0, 590, 12)
    on_line = [L.goodman_sum(L.SE * (1 - s / L.SU), s) for s in sms]
    checks.append(("Goodman line: S_a/S_e + S_m/S_u = 1 on 12 points of the line", bool(np.allclose(on_line, 1.0))))
    eq = [L.equivalent_amplitude(L.SE * (1 - s / L.SU), s) for s in sms]
    checks.append(("every point on the Goodman line has equivalent amplitude S_e = 300 MPa", bool(np.allclose(eq, L.SE))))
    checks.append(("a point inside the line, (100, 150), has sum 0.583 and infinite life", abs(L.goodman_sum(100, 150) - 0.5833) < 1e-3 and L.life_with_mean(100, 150) == np.inf))
    n_a = L.life_with_mean(200, 300)
    checks.append((f"(S_a, S_m) = (200, 300) has sum 1.167 and equals a 400 MPa reversed load: about 34,000 cycles (got {n_a:,.0f})",
                   abs(L.goodman_sum(200, 300) - 1.1667) < 1e-3 and abs(L.equivalent_amplitude(200, 300) - 400) < 1e-9 and abs(n_a - 34017) < 5))
    checks.append(("a tensile mean stress shortens life", L.life_with_mean(250, 100) < L.life_with_mean(250, 0) or L.life_with_mean(250, 0) == np.inf and L.life_with_mean(340, 100) < L.life_with_mean(340, 0)))
    try:
        L.equivalent_amplitude(100, 600)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge: mean stress equal to S_u is rejected", raised))

    # stress concentration
    kt = L.stress_concentration_hole()
    checks.append((f"K_t of a hole in a wide plate is 3 (got {kt:.4f})", abs(kt - 3.0) < 1e-6))
    checks.append(("hole: stress at 90 degrees is 3 and at 0 degrees is -1 (compression)", abs(L.hole_hoop_stress(90) - 3) < 1e-12 and abs(L.hole_hoop_stress(0) + 1) < 1e-12))
    checks.append(("hole: far from the hole the stress approaches the nominal value 1 (at 0 and 90 degrees mean)", abs(L.hole_hoop_stress(90, 50.0) - 1.0) < 2e-3))
    n_hole = L.life_with_notch(100, 50, 3.0)
    checks.append((f"plate with a hole, nominal (100, 50) MPa: about 34,000 cycles; without the hole infinite (got {n_hole:,.0f})",
                   abs(n_hole - 34017) < 5 and L.life_with_notch(100, 50, 1.0) == np.inf))

    # worked example: shaft
    d = L.design_diameter_mm(200, 2.0, 1.0)
    s27 = L.bending_stress_mpa(200, 0.027)
    hand = (32 * 200 * 2 / (np.pi * 210e6)) ** (1 / 3) * 1000
    checks.append((f"worked example: shaft M = 200 N m, n = 2 needs d = 26.87 mm (got {d:.2f}); take 27 mm", abs(d - 26.87) < 0.01 and abs(d - hand) < 1e-9))
    checks.append((f"worked example: 27 mm gives 103.5 MPa and margin 2.03 (got {s27:.1f}, {210 / s27:.2f})", abs(s27 - 103.5) < 0.05 and abs(210 / s27 - 2.03) < 0.01))
    checks.append(("worked example: with a shoulder K_t = 2 the diameter becomes 33.86 mm", abs(L.design_diameter_mm(200, 2.0, 2.0) - 33.86) < 0.01))
    checks.append(("27 mm shaft has indefinite life in this model", L.shaft_life(200, 27) == np.inf))
    n20 = L.shaft_life(200, 20)
    checks.append((f"a 20 mm shaft (254.6 MPa, above 210) lasts about 244,000 cycles (got {n20:,.0f})", abs(L.bending_stress_mpa(200, 0.02) - 254.6) < 0.1 and abs(n20 - 244157) < 20))
    checks.append(("design diameter grows with moment, safety factor and K_t",
                   L.design_diameter_mm(300) > L.design_diameter_mm(200) and L.design_diameter_mm(200, 3) > L.design_diameter_mm(200, 2) and L.design_diameter_mm(200, 2, 2) > L.design_diameter_mm(200, 2, 1)))
    checks.append(("Section F: M = 150 N m, n = 2, K_t = 1.5 gives 27.95 mm; with K_t = 1 it is 24.41 mm",
                   abs(L.design_diameter_mm(150, 2, 1.5) - 27.95) < 0.01 and abs(L.design_diameter_mm(150, 2, 1.0) - 24.41) < 0.01))
    s28 = 1.5 * L.bending_stress_mpa(150, 0.028)
    checks.append((f"Section F: 28 mm with K_t = 1.5 gives local stress 104.4 MPa (got {s28:.1f}); 22 mm fails after about 835,000 cycles",
                   abs(s28 - 104.4) < 0.05 and abs(L.shaft_life(150, 22, 1.5) - 835170) < 50))
    # extra numbers used in the text
    checks.append(("1500 revolutions per minute is 25 Hz, so 1 million cycles take 11.1 hours", abs(1e6 / (1500 / 60.0) / 3600 - 11.11) < 0.01))
    checks.append(("S-N values: 365 MPa gives 100,000 cycles and 444 MPa gives 10,000 cycles", abs(L.stress_for_life(1e5) - 364.9) < 0.1 and abs(L.stress_for_life(1e4) - 443.9) < 0.1))
    checks.append((f"Section F: (S_a, S_m) = (240, 250) has S_eq = 411.4 MPa and life about 24,400 cycles (got {L.life_with_mean(240, 250):,.0f})",
                   abs(L.equivalent_amplitude(240, 250) - 411.4) < 0.05 and abs(L.life_with_mean(240, 250) - 24430) < 10))
    checks.append(("Section F: (S_a, S_m) = (150, 300) lies exactly on the Goodman line (sum 1.000)", abs(L.goodman_sum(150, 300) - 1.0) < 1e-12))
    m25 = 210 / (1.5 * L.bending_stress_mpa(150, 0.025))
    m28 = 210 / (1.5 * L.bending_stress_mpa(150, 0.028))
    checks.append((f"Section F: margins for M = 150 N m, K_t = 1.5: 25 mm gives {m25:.2f} (too low), 28 mm gives {m28:.2f}", abs(m25 - 1.43) < 0.005 and abs(m28 - 2.01) < 0.005))
    checks.append(("Section F: hole plate with nominal amplitude 80 MPa has infinite life, with 110 MPa about 326,000 cycles",
                   L.life_with_notch(80, 0, 3) == np.inf and abs(L.life_with_notch(110, 0, 3) - 326247) < 20))
    checks.append(("challenge: K_t = 3 with nominal (60, 30) MPa gives Goodman sum 0.75 and infinite life", abs(L.goodman_sum(180, 90) - 0.75) < 1e-12 and L.life_with_notch(60, 30, 3) == np.inf))
    checks.append(("a shoulder with K_t = 2 costs about 7 mm of diameter (33.86 - 26.87 = 6.99)", abs((L.design_diameter_mm(200, 2, 2) - L.design_diameter_mm(200, 2, 1)) - 6.99) < 0.01 and abs(L.design_diameter_mm(200, 2, 2) / L.design_diameter_mm(200, 2, 1) - 2 ** (1 / 3)) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
